# Olist E-commerce RFM & Customer Retention Analysis

**Goal:** analyze delivered orders, perform EDA, segment customers using Recency/Frequency/Monetary (RFM), and inspect repeat purchasing and monthly cohorts.

**Dataset:** [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)

Run every cell in order. Upload the dataset ZIP when prompted. The raw dataset is not uploaded to GitHub.

## 1. Install dependencies

In [ ]:
%pip -q install duckdb pandas numpy matplotlib seaborn


## 2. Imports and output folders

In [ ]:
from pathlib import Path
import zipfile
import warnings

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")

PROJECT_DIR = Path("/content/olist-rfm-customer-retention")
DATA_DIR = Path("/content/olist_data")
CHART_DIR = PROJECT_DIR / "outputs" / "charts"
SQL_DIR = PROJECT_DIR / "sql"

for folder in (DATA_DIR, CHART_DIR, SQL_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print("DuckDB version:", duckdb.__version__)
print("Pandas version:", pd.__version__)


## 3. Upload and extract the dataset ZIP

Download the Olist dataset ZIP from Kaggle, then select it in the upload dialog. If your CSV files are already in the Colab runtime, you can skip the upload and extraction cell after setting `DATA_DIR` to their folder.

In [ ]:
from google.colab import files

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]

if zip_names:
    zip_path = Path("/content") / zip_names[0]
    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall(DATA_DIR)
    print(f"Extracted {zip_names[0]} into {DATA_DIR}")
else:
    print("No ZIP uploaded. Make sure the dataset CSV files are available under", DATA_DIR)

csv_paths = sorted(DATA_DIR.rglob("*.csv"))
print(f"CSV files found: {len(csv_paths)}")
for path in csv_paths:
    print("-", path.name)


## 4. Load the core tables

The notebook requires orders, customers, and order items. Other tables are optional for this version.

In [ ]:
def find_csv(filename: str) -> Path:
    matches = list(DATA_DIR.rglob(filename))
    if not matches:
        raise FileNotFoundError(
            f"Could not find {filename}. Check that the Olist ZIP was extracted into {DATA_DIR}."
        )
    return matches[0]

orders = pd.read_csv(
    find_csv("olist_orders_dataset.csv"),
    parse_dates=["order_purchase_timestamp"],
)
customers = pd.read_csv(find_csv("olist_customers_dataset.csv"))
order_items = pd.read_csv(find_csv("olist_order_items_dataset.csv"))

print("Orders:", orders.shape)
print("Customers:", customers.shape)
print("Order items:", order_items.shape)

display(orders.head())
display(customers.head())
display(order_items.head())


## 5. Data quality checks

In [ ]:
def quality_report(df: pd.DataFrame, name: str) -> pd.DataFrame:
    report = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_count": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "unique_values": df.nunique(dropna=True),
    })
    print(f"{name}: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print("Exact duplicate rows:", int(df.duplicated().sum()))
    display(report.sort_values("missing_count", ascending=False))
    return report

orders_quality = quality_report(orders, "orders")
customers_quality = quality_report(customers, "customers")
items_quality = quality_report(order_items, "order_items")

print("Duplicate order IDs:", int(orders["order_id"].duplicated().sum()))
print("Duplicate customer_id records:", int(customers["customer_id"].duplicated().sum()))
print("\nOrder status counts:")
display(orders["order_status"].value_counts(dropna=False).to_frame("orders"))


## 6. Basic exploratory data analysis

In [ ]:
# Order status distribution
status_counts = orders["order_status"].value_counts(dropna=False)
plt.figure(figsize=(9, 4))
sns.barplot(x=status_counts.index.astype(str), y=status_counts.values)
plt.title("Orders by status")
plt.xlabel("Order status")
plt.ylabel("Number of orders")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(CHART_DIR / "orders_by_status.png", dpi=160, bbox_inches="tight")
plt.show()

# Orders by purchase month
monthly_orders = (
    orders.dropna(subset=["order_purchase_timestamp"])
    .assign(order_month=lambda d: d["order_purchase_timestamp"].dt.to_period("M").dt.to_timestamp())
    .groupby("order_month")["order_id"]
    .nunique()
    .rename("orders")
    .reset_index()
)
plt.figure(figsize=(12, 4))
sns.lineplot(data=monthly_orders, x="order_month", y="orders", marker="o")
plt.title("Monthly order volume")
plt.xlabel("Purchase month")
plt.ylabel("Distinct orders")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(CHART_DIR / "monthly_order_volume.png", dpi=160, bbox_inches="tight")
plt.show()

display(monthly_orders.tail(12))


## 7. Build DuckDB tables and an order-level value view

We use **delivered orders only** for RFM. Monetary value is the sum of item prices, excluding freight. We aggregate items to one row per order before customer-level analysis to avoid double-counting customers or orders.

In [ ]:
con = duckdb.connect(database=":memory:")
con.register("orders_df", orders)
con.register("customers_df", customers)
con.register("order_items_df", order_items)

con.execute("CREATE OR REPLACE TABLE orders AS SELECT * FROM orders_df")
con.execute("CREATE OR REPLACE TABLE customers AS SELECT * FROM customers_df")
con.execute("CREATE OR REPLACE TABLE order_items AS SELECT * FROM order_items_df")

con.execute("""
CREATE OR REPLACE TEMP VIEW delivered_order_value AS
SELECT
    o.order_id,
    c.customer_unique_id,
    CAST(o.order_purchase_timestamp AS TIMESTAMP) AS purchase_ts,
    SUM(COALESCE(oi.price, 0)) AS order_monetary_value
FROM orders AS o
JOIN customers AS c ON o.customer_id = c.customer_id
JOIN order_items AS oi ON o.order_id = oi.order_id
WHERE o.order_status = 'delivered'
  AND o.order_purchase_timestamp IS NOT NULL
GROUP BY o.order_id, c.customer_unique_id, CAST(o.order_purchase_timestamp AS TIMESTAMP)
""")

print("Delivered order-value rows:", con.execute("SELECT COUNT(*) FROM delivered_order_value").fetchone()[0])
display(con.execute("SELECT * FROM delivered_order_value LIMIT 5").df())


## 8. RFM analysis using SQL

- **Recency:** days since the customer's latest delivered order.
- **Frequency:** number of distinct delivered orders.
- **Monetary:** total item-price value across delivered orders.

Quantile scores range from 1 to 5. For Recency, fewer days means a higher score. The segment names are practical heuristics, not ground-truth labels.

In [ ]:
rfm_sql = """
CREATE OR REPLACE TEMP VIEW rfm_base AS
WITH reference AS (
    SELECT CAST(MAX(purchase_ts) + INTERVAL 1 DAY AS TIMESTAMP) AS reference_date
    FROM delivered_order_value
)
SELECT
    d.customer_unique_id,
    DATE_DIFF('day', MAX(d.purchase_ts), r.reference_date) AS recency_days,
    COUNT(DISTINCT d.order_id) AS frequency,
    SUM(d.order_monetary_value) AS monetary,
    MAX(d.purchase_ts) AS last_purchase_ts,
    MIN(d.purchase_ts) AS first_purchase_ts
FROM delivered_order_value d
CROSS JOIN reference r
GROUP BY d.customer_unique_id, r.reference_date;

CREATE OR REPLACE TEMP VIEW rfm_scored AS
WITH scored AS (
    SELECT *,
        NTILE(5) OVER (ORDER BY recency_days DESC) AS r_score,
        NTILE(5) OVER (ORDER BY frequency ASC) AS f_score,
        NTILE(5) OVER (ORDER BY monetary ASC) AS m_score
    FROM rfm_base
)
SELECT *,
    CAST(r_score AS VARCHAR) || CAST(f_score AS VARCHAR) || CAST(m_score AS VARCHAR) AS rfm_score,
    CASE
        WHEN r_score >= 4 AND f_score >= 4 AND m_score >= 4 THEN 'Champions'
        WHEN r_score >= 3 AND f_score >= 4 THEN 'Loyal Customers'
        WHEN r_score >= 4 AND f_score BETWEEN 2 AND 3 THEN 'Recent Customers'
        WHEN r_score <= 2 AND f_score >= 3 THEN 'Needs Attention'
        WHEN r_score <= 2 AND f_score <= 2 THEN 'At Risk'
        ELSE 'Potential Loyalists'
    END AS customer_segment
FROM scored;
"""
con.execute(rfm_sql)

rfm = con.execute("SELECT * FROM rfm_scored ORDER BY monetary DESC").df()
display(rfm.head(10))
print("Customers in RFM table:", len(rfm))
print("Reference date:", (pd.to_datetime(rfm["last_purchase_ts"]).max() + pd.Timedelta(days=1)).date())


## 9. Visualize RFM segments and distributions

In [ ]:
segment_counts = (
    rfm["customer_segment"].value_counts()
    .rename_axis("customer_segment")
    .reset_index(name="customers")
)
display(segment_counts)

plt.figure(figsize=(10, 5))
sns.barplot(data=segment_counts, x="customer_segment", y="customers")
plt.title("Customer count by RFM segment")
plt.xlabel("RFM segment")
plt.ylabel("Customers")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.savefig(CHART_DIR / "rfm_segment_counts.png", dpi=160, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(rfm["recency_days"], bins=40, ax=axes[0])
axes[0].set_title("Recency (days)")
sns.histplot(rfm["frequency"], bins=min(30, max(5, int(rfm["frequency"].nunique()))), ax=axes[1])
axes[1].set_title("Delivered order frequency")
sns.histplot(rfm["monetary"], bins=40, ax=axes[2])
axes[2].set_title("Monetary value (item prices)")
for ax in axes:
    ax.set_ylabel("Customers")
plt.tight_layout()
plt.savefig(CHART_DIR / "rfm_distributions.png", dpi=160, bbox_inches="tight")
plt.show()


## 10. Repeat-purchase analysis

In [ ]:
customer_order_counts = (
    con.execute("""
        SELECT customer_unique_id, COUNT(DISTINCT order_id) AS delivered_orders
        FROM delivered_order_value
        GROUP BY customer_unique_id
    """).df()
)
repeat_summary = pd.DataFrame({
    "metric": [
        "Customers with delivered orders",
        "Customers with 2+ delivered orders",
        "Repeat-customer percentage",
        "Average delivered orders per customer",
    ],
    "value": [
        len(customer_order_counts),
        int((customer_order_counts["delivered_orders"] >= 2).sum()),
        round(100 * (customer_order_counts["delivered_orders"] >= 2).mean(), 2),
        round(customer_order_counts["delivered_orders"].mean(), 2),
    ],
})
display(repeat_summary)

plt.figure(figsize=(8, 4))
sns.countplot(data=customer_order_counts, x="delivered_orders")
plt.title("Delivered orders per customer")
plt.xlabel("Number of delivered orders")
plt.ylabel("Customers")
plt.tight_layout()
plt.savefig(CHART_DIR / "orders_per_customer.png", dpi=160, bbox_inches="tight")
plt.show()


## 11. Monthly cohort retention

Each customer's cohort is the month of their first observed delivered order. A customer is active in a month if they made at least one delivered order in that month. Later cohort months may be incomplete because the dataset has a finite end date.

In [ ]:
cohort_activity = con.execute("""
WITH customer_months AS (
    SELECT DISTINCT
        customer_unique_id,
        CAST(DATE_TRUNC('month', purchase_ts) AS DATE) AS order_month
    FROM delivered_order_value
),
customer_cohorts AS (
    SELECT customer_unique_id, MIN(order_month) AS cohort_month
    FROM customer_months
    GROUP BY customer_unique_id
)
SELECT
    cm.customer_unique_id,
    cc.cohort_month,
    cm.order_month,
    (
      (DATE_PART('year', cm.order_month) - DATE_PART('year', cc.cohort_month)) * 12
      + (DATE_PART('month', cm.order_month) - DATE_PART('month', cc.cohort_month))
    )::INTEGER AS cohort_index
FROM customer_months cm
JOIN customer_cohorts cc USING (customer_unique_id)
ORDER BY cohort_month, cohort_index
""").df()

cohort_counts = (
    cohort_activity.groupby(["cohort_month", "cohort_index"])["customer_unique_id"]
    .nunique()
    .rename("active_customers")
    .reset_index()
)
cohort_sizes = (
    cohort_counts[cohort_counts["cohort_index"] == 0][["cohort_month", "active_customers"]]
    .rename(columns={"active_customers": "cohort_size"})
)
cohort_counts = cohort_counts.merge(cohort_sizes, on="cohort_month", how="left")
cohort_counts["retention_rate"] = cohort_counts["active_customers"] / cohort_counts["cohort_size"]

retention_pivot = cohort_counts.pivot(
    index="cohort_month", columns="cohort_index", values="retention_rate"
).sort_index()

display(retention_pivot.tail(12).style.format("{:.0%}"))

plt.figure(figsize=(14, 8))
sns.heatmap(retention_pivot, annot=False, fmt=".0%", cmap="Blues", vmin=0, vmax=1)
plt.title("Monthly customer cohort retention")
plt.xlabel("Months since first delivered order")
plt.ylabel("First delivered-order month")
plt.tight_layout()
plt.savefig(CHART_DIR / "cohort_retention_heatmap.png", dpi=160, bbox_inches="tight")
plt.show()


## 12. Export analysis outputs

In [ ]:
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

rfm.to_csv(OUTPUT_DIR / "rfm_customer_segments.csv", index=False)
segment_counts.to_csv(OUTPUT_DIR / "rfm_segment_counts.csv", index=False)
repeat_summary.to_csv(OUTPUT_DIR / "repeat_purchase_summary.csv", index=False)
cohort_counts.to_csv(OUTPUT_DIR / "cohort_retention_long.csv", index=False)
retention_pivot.to_csv(OUTPUT_DIR / "cohort_retention_matrix.csv")

print("Saved tables to:", OUTPUT_DIR)
print("Saved charts to:", CHART_DIR)
print("Charts:")
for path in sorted(CHART_DIR.glob("*.png")):
    print("-", path.name)


## 13. Write findings after inspecting your results

Use the actual output tables and charts to complete this section. Do not report guessed values.

Suggested questions:
1. Which RFM segment contains the most customers?
2. What percentage of customers made at least two delivered purchases?
3. Which cohort months show the strongest later-month retention?
4. How does order volume change over the observed time period?
5. What data limitations could affect interpretation?

## Limitations
- The dataset reflects a particular marketplace, geography, and historical period.
- Only delivered orders with matching customer and order-item records are included in RFM.
- Monetary value excludes freight and does not account for refunds, discounts, or profit.
- Cohort retention measures observed delivered-order activity, not a guarantee of future behavior.
- RFM segment labels are heuristic business interpretations, not validated customer labels.

## 14. Optional: download chart outputs

To download a ZIP of charts from Colab, run this cell.

In [ ]:
import shutil
from google.colab import files

charts_zip = shutil.make_archive("/content/olist_charts", "zip", CHART_DIR)
files.download(charts_zip)
